# Similarity Search with Embeddings

## Introduction

Satellite embeddings are compact, high-dimensional vector representations of satellite imagery generated by a deep learning model. Instead of raw pixel values, each location is represented by a dense vector that captures the semantic content of the landscape — making it possible to search for visually similar areas using vector operations. [AlphaEarth Embeddings (AEF)](https://aef-loader.readthedocs.io/en/latest/) is an openly available global dataset of satellite embeddings derived from multiple earth observation datasets, accessible via [Source Cooperative](https://source.coop/). The [`aef-loader`](https://aef-loader.readthedocs.io/en/latest/) package provides a convenient Python interface to query and stream these embeddings without downloading the entire dataset.

This tutorial is an open-source adaptation of our [Google Earth Engine community tutorial on Satellite Embedding Similarity Search](https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-05-similarity-search). We replicate the same workflow using open-access data and open-source packages — `xarray`, `dask`, and `scikit-learn`.

## Overview of the Task

We will use AlphaEarth Embeddings to perform a **similarity search** for grain silos in Franklin County, Kansas, USA. Starting from a few known silo locations, we compute a reference embedding vector by averaging the embeddings at those points. We then compute the cosine similarity between this reference vector and every pixel in the county, and threshold the result to identify other areas with grain silos.

<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/Grain_bins_with_dryer_unit.jpg/330px-Grain_bins_with_dryer_unit.jpg" alt="Grain Silos">
</p>
<p align="center"><em>Grain silos (image: Wikipedia)</em></p>


**Input Layers**:

* AlphaEarth Embeddings (AEF) for year 2024, streamed via the `aef-loader` package.
* `cb_2021_us_county_500k.zip`: US county boundaries from the US Census Bureau, used to define the study area (Franklin County, Kansas).

**Output**:

* `cosine_similarity.tif`: A GeoTIFF raster of per-pixel cosine similarity scores relative to the reference grain silo embedding.
* `similar_pixels.gpkg`: A GeoPackage of vectorized polygons for pixels that exceed the similarity threshold.

**Data Credit**:

* AlphaEarth Foundations (AEF) Satellite Embeddings : The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind. Accessed from [Source Cooperative](https://source.coop/tge-labs/aef).
* County boundary data: US Census Bureau, 2021 Cartographic Boundary Files.

## Setup and Data Download
The following blocks of code will install the required packages and download the datasets to your Colab environment.

In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip install rioxarray aef-loader dask[distributed] leafmap

In [ ]:
import asyncio
import os
import dask.array as da
import geopandas as gpd
import leafmap.foliumap as leafmap
import numpy as np
import rioxarray as rxr
import xarray as xr
from aef_loader import AEFIndex, VirtualTiffReader, DataSource
from aef_loader.utils import dequantize_aef, reproject_datatree
from leafmap.basemaps import GoogleMapsTileProvider
from odc.geo.geobox import GeoBox
from pyproj import Transformer
from rasterio.features import shapes
from shapely.geometry import shape
from sklearn.metrics.pairwise import cosine_similarity

Setup a local Dask cluster. This distributes the computation across multiple workers on your computer.

In [ ]:
from dask.distributed import Client
client = Client()  # set up local cluster on the machine
client

If you are running this notebook in Colab, you will need to create and use a proxy URL to see the dashboard running on the local server.



In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import output
    port_to_expose = 8787  # This is the default port for Dask dashboard
    print(output.eval_js(f'google.colab.kernel.proxyPort({port_to_expose})'))

In [ ]:
data_folder = 'data'
output_folder = 'output'

if not os.path.exists(data_folder):
    os.mkdir(data_folder)
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

In [ ]:
def download(url):
    filename = os.path.join(data_folder, os.path.basename(url))
    if not os.path.exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)

counties_file = 'cb_2021_us_county_500k.zip'

download('https://www2.census.gov/geo/tiger/GENZ2021/shp/' + counties_file)

## Select the search region

For this tutorial, we will map the grain silos in Franklin County, Kansas. First we will read the Zipped counties shapefile and apply a filter and select the polygon for this county.

In [31]:
counties_file_path = os.path.join(data_folder, counties_file)

counties_df = gpd.read_file(counties_file_path)
selected = counties_df[counties_df['GEOID'] == '20059']  # Franklin County, Kansas
selected.iloc[:, :6]

,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME
2730,20,059,00484998,0500000US20059,20059,Franklin


In [30]:
m = leafmap.Map(width=600, height=500)
m.add_tile_layer(
    url='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    name='Google Satellite',
    attribution='Google',
)
m.add_gdf(selected, layer_name="Selected County")
m.zoom_to_gdf(selected)
m

In [32]:
bbox = list(selected.total_bounds)
year = 2024
target_crs = 'EPSG:3857'

# Transform bbox from EPSG:4326 to chosen crs
transformer = Transformer.from_crs('EPSG:4326', target_crs, always_xy=True)
x_min, y_min = transformer.transform(bbox[0], bbox[1])
x_max, y_max = transformer.transform(bbox[2], bbox[3])

target = GeoBox.from_bbox(
    bbox=(x_min, y_min, x_max, y_max),
    crs=target_crs,
    resolution=10,
)

m = leafmap.Map(width=600, height=500)
m.add_tile_layer(
    url='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    name='Google Satellite',
    attribution='Google',
)
target.explore(map=m)
m

In [33]:
index = AEFIndex(source=DataSource.SOURCE_COOP)
await index.download()

# Query for tiles
tiles = await index.query(
    bbox=bbox,
    years=(year),
)
# Load tiles organized by UTM zone
async with VirtualTiffReader() as reader:
    tree = await reader.open_tiles_by_zone(tiles)

combined = reproject_datatree(tree, target)
embeddings = combined.embeddings
embeddings

<xarray.DataArray 'embeddings' (time: 1, band: 64, y: 4975, x: 5039)> Size: 2GB
dask.array<reproject, shape=(1, 64, 4975, 5039), dtype=int8, chunksize=(1, 1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 8B 2024-01-01
  * band         (band) <U3 768B 'A00' 'A01' 'A02' 'A03' ... 'A61' 'A62' 'A63'
  * y            (y) float64 40kB 4.684e+06 4.684e+06 ... 4.635e+06 4.635e+06
  * x            (x) float64 40kB -1.063e+07 -1.063e+07 ... -1.058e+07
    spatial_ref  int32 4B 3857
Attributes:
    citation:                    True
    geog_angular_units:          True
    geog_citation:               True
    model_type:                  True
    proj_linear_units:           True
    projected_type:              True
    raster_type:                 True
    photometric_interpretation:  1
    DESCRIPTION:                 A63
    gdal_no_data:                -128
    nodata:                      -128
    _FillValue:                  -128

In [34]:
embeddings_year = embeddings.isel(time=0)
embeddings_float = dequantize_aef(embeddings_year)
embeddings_float

<xarray.DataArray 'embeddings' (band: 64, y: 4975, x: 5039)> Size: 6GB
dask.array<where, shape=(64, 4975, 5039), dtype=float32, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) <U3 768B 'A00' 'A01' 'A02' 'A03' ... 'A61' 'A62' 'A63'
  * y            (y) float64 40kB 4.684e+06 4.684e+06 ... 4.635e+06 4.635e+06
  * x            (x) float64 40kB -1.063e+07 -1.063e+07 ... -1.058e+07
    time         datetime64[ns] 8B 2024-01-01
    spatial_ref  int32 4B 3857
Attributes: (12/14)
    citation:                    True
    geog_angular_units:          True
    geog_citation:               True
    model_type:                  True
    proj_linear_units:           True
    projected_type:              True
    ...                          ...
    DESCRIPTION:                 A63
    gdal_no_data:                -128
    nodata:                      nan
    _FillValue:                  nan
    units:                       embedding
    dequantized:                 True

In [35]:
target_location1 = (-95.18616479385629, 38.54715519758577)
target_location2 = (-95.34468619878159, 38.59339901996762)
target_location3 = (-95.34280239688128, 38.56233059960432)

m = leafmap.Map(width=600, height=500)
m.add_tile_layer(
    url='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    name='Google Satellite',
    attribution='Google',
)
for i, loc in enumerate([target_location1, target_location2, target_location3], 1):
    m.add_marker(location=(loc[1], loc[0]), popup=f'Target {i}')
m.zoom_to_gdf(selected)

m

In [ ]:
# Extract embeddings from all 3 target locations and compute mean (lazy)
target_locations = [target_location1, target_location2, target_location3]

target_embeddings = []
for lon, lat in target_locations:
    tx, ty = transformer.transform(lon, lat)
    emb_at_point = embeddings_float.sel(x=tx, y=ty, method="nearest").data  # stays as dask
    target_embeddings.append(emb_at_point)

mean_embedding = da.mean(da.stack(target_embeddings, axis=0), axis=0)

In [ ]:
%%time
mean_embedding = mean_embedding.compute()

In [ ]:
# Get embeddings as dask array: (bands, y, x)
emb = embeddings_float.data
# Transpose to (y, x, bands) then reshape to (y*x, bands)
emb = da.moveaxis(emb, 0, -1)  # (y, x, bands)
ny, nx, nb = emb.shape
emb_2d = emb.reshape(-1, nb)  # (y*x, bands)

# Rechunk so each block has ALL bands (axis 1 = single chunk)
emb_2d_rechunked = emb_2d.rechunk({1: -1})
emb_2d_rechunked

In [ ]:
# Compute cosine similarity between the target embedding and every pixel
# mean_embedding is 1-D (nb,); emb_2d is (ny*nx, nb)

# Bring target embedding to a 2-D numpy array (1, nb)
target_vec = mean_embedding.reshape(1, -1)

# Compute in chunks to stay memory-friendly
def cosine_sim_block(block, target=target_vec):
    """Compute cosine similarity for one chunk of pixels."""
    return cosine_similarity(block, target).ravel()  # (chunk_size,)

# Map over dask blocks – each block is (chunk_pixels, nb) → (chunk_pixels,)
sim_1d = emb_2d_rechunked.map_blocks(
    cosine_sim_block,
    dtype=np.float64,
    drop_axis=1,  # the bands axis is collapsed
)
sim_1d

In [ ]:
%%time
sim_values = sim_1d.compute()

In [ ]:
# Build an xarray DataArray with the same spatial coords as the input

similarity = xr.DataArray(
    sim_values.reshape(ny, nx), #reshape back to (ny, nx)
    dims=["y", "x"],
    coords={
        "y": embeddings.coords["y"].values,
        "x": embeddings.coords["x"].values,
    },
    name="cosine_similarity",
)

# Copy CRS and spatial metadata from the original embeddings
similarity = similarity.rio.write_crs(embeddings_float.rio.crs)
similarity = similarity.rio.write_transform(embeddings_float.rio.transform())
similarity

In [ ]:
# Apply threshold and convert matching pixels to polygons
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd

threshold = 0.95

# Create a binary mask: 1 where similarity >= threshold, 0 elsewhere
mask = (similarity.values >= threshold).astype(np.uint8)

# Get the affine transform from the similarity raster
transform = similarity.rio.transform()
crs = similarity.rio.crs

# Vectorize: convert raster mask to polygon geometries
polygons = []
values = []
for geom, val in shapes(mask, mask=(mask == 1), transform=transform):
    polygons.append(shape(geom))
    values.append(val)

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame(
    {"similarity": values},
    geometry=polygons,
    crs=crs,
)

gdf.head()

In [36]:
m = leafmap.Map(width=600, height=500)
m.add_tile_layer(
    url='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    name='Google Satellite',
    attribution='Google',
)

for i, loc in enumerate([target_location1, target_location2, target_location3], 1):
    m.add_marker(location=(loc[1], loc[0]), popup=f'Target {i}')
m.zoom_to_gdf(selected)

m.add_gdf(gdf, layer_name="Similar Areas",
          style={'color': 'red', 'fillColor': 'red', 'fillOpacity': 0.5})
m

In [ ]:
# Save to GeoTIFF
output_path = os.path.join(output_folder, 'cosine_similarity.tif')
similarity.rio.to_raster(output_path)
print(f'Saved similarity raster to {output_path}')

# Save to GeoPackage
vector_path = os.path.join(output_folder, 'predicted_matches.gpkg')
gdf.to_file(vector_path, driver='GPKG')
print(f'Saved predicted matches to {vector_path}')